# UIT-ViQuAD2.0 — Qwen2.5-3B: Export `results.json` (key = uit_id)

Notebook **chỉ để suy luận + xuất submission**, không train.

Luồng:
1. **Auto-detect** các thư mục adapter trong folder hiện tại:
   `qwen2.5-3b-instruct-lora-viquad2`, `...-dora-viquad2`, `...-tinylora-viquad2`
2. Load **full test split (~7301)** từ HuggingFace (`taidng/UIT-ViQuAD2.0`) kèm **`uit_id`**
3. Với mỗi adapter tồn tại → chạy inference → ghi **`results.json`** dạng `{uit_id: answer}` vào chính thư mục adapter đó

> Pipeline prompt / postprocess / no-answer sentinel **giống hệt** notebook 1.5B để kết quả nhất quán.

### Cách chạy
1. (Nếu cần) chạy cell pip → **Restart Kernel**
2. Chạy lần lượt: Imports → Config & Data → Inference funcs → **Export**
3. Chỉ muốn 1 loại: sửa `EXPORT_METHODS = ["dora"]`

In [ ]:
# (Tùy chọn) cài môi trường — bỏ qua nếu kernel đã có sẵn Unsloth/PEFT
# !pip install "peft>=0.19.0" transformers trl accelerate bitsandbytes xformers datasets --no-cache-dir
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
# !pip install unsloth_zoo scikit-learn --no-cache-dir
print("Bỏ qua pip nếu môi trường đã sẵn sàng.")

In [ ]:
import warnings, logging, os, sys
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["HF_HUB_DISABLE_XET"] = "1"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass
print("Warnings suppressed.")

In [ ]:
import gc
import json
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# ========== CONFIG (đồng bộ với notebook 1.5B) ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # chỉ là fallback; base thật đọc từ adapter_config.json
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)
USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn. Chỉ trả về cụm từ trong đoạn văn.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

# Adapter folders (theo naming 3B). Notebook sẽ tự bỏ qua folder không tồn tại.
ADAPTER_SUBMISSION_DIRS = {
    "lora": "qwen2.5-3b-instruct-lora-viquad2",
    "dora": "qwen2.5-3b-instruct-dora-viquad2",
    "tinylora": "qwen2.5-3b-instruct-tinylora-viquad2",
}
# Chọn method muốn export (chỉ những cái có folder + adapter mới chạy)
EXPORT_METHODS = ["lora", "dora", "tinylora"]

LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
MAX_NEW_TOKENS = 32
USE_SPAN_POSTPROCESS = True
USE_UNSLOTH_FOR_INFERENCE = False

# max_seq_length inference: đọc profiling nếu có, không thì 1024
_prof = Path("profiling_config_3b.json")
if not _prof.exists():
    _prof = Path("profiling_config.json")
if _prof.exists():
    MAX_SEQ_LENGTH = json.load(open(_prof, encoding="utf-8"))["max_seq_length"]
    print(f"max_seq_length={MAX_SEQ_LENGTH} (from {_prof.name})")
else:
    MAX_SEQ_LENGTH = 1024
    print(f"max_seq_length={MAX_SEQ_LENGTH} (default)")

SUBMISSION_MAX_SAMPLES = None   # None = full 7301; số nhỏ để debug nhanh
SUBMISSION_LOG_EVERY = 50
TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
# ========== Load full test split (kèm uit_id) ==========
def row_to_sample(row):
    is_impossible = bool(row.get("is_impossible", False))
    answers = row.get("answers")
    answer = None
    has_label = False
    if answers is not None:
        texts = answers.get("text") or []
        if is_impossible or len(texts) == 0 or not str(texts[0]).strip():
            answer, is_impossible, has_label = NO_ANSWER_SENTINEL, True, True
        else:
            answer, is_impossible, has_label = texts[0], False, True
    return {
        "id": row.get("id", ""),
        "uit_id": row.get("uit_id", ""),
        "title": row.get("title", ""),
        "context": row["context"],
        "question": row["question"],
        "answer": answer,
        "is_impossible": is_impossible,
        "has_label": has_label,
    }


raw_test = load_dataset(DATASET_NAME, split="test")
if "uit_id" not in raw_test.column_names:
    raise RuntimeError(f"Test split thiếu cột uit_id. Cột hiện có: {raw_test.column_names}")

test_samples = [row_to_sample(r) for r in raw_test]

missing_uit = sum(1 for s in test_samples if not s["uit_id"])
n_no = sum(1 for s in test_samples if s["is_impossible"])
print(f"Test samples: {len(test_samples)} | NoAns(label)={n_no} | thiếu uit_id={missing_uit}")
print("Ví dụ:", {k: test_samples[0][k] for k in ("id", "uit_id", "question")})
if missing_uit:
    raise RuntimeError(f"{missing_uit} mẫu thiếu uit_id — không thể xuất submission chuẩn.")

In [ ]:
# ========== Postprocess + no-answer + span align (giống 1.5B) ==========
PREFIX_RE = re.compile(
    r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*",
    re.IGNORECASE,
)


def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())


def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt:
        return 1.0
    if not pt or not tt:
        return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if n == 0:
        return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)


def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)


def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip('"\'')
    return PREFIX_RE.sub("", pred).strip()


def align_to_context(pred, context):
    if not pred or is_no_answer(pred):
        return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0:
        return context[idx:idx + len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words:
        return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i + 1, min(i + n + 4, len(words)) + 1):
            span = " ".join(words[i:j])
            f1 = compute_f1(span, pred)
            if f1 > best_f1:
                best_f1, best_span = f1, span
    return best_span.strip() if best_f1 >= 0.5 else pred

In [ ]:
# ========== Load adapter + inference ==========
def _read_adapter_base_model(adapter_path):
    cfg_path = Path(adapter_path) / "adapter_config.json"
    if not cfg_path.exists():
        raise FileNotFoundError(f"Thiếu adapter_config.json trong {adapter_path}")
    cfg = json.load(open(cfg_path, encoding="utf-8"))
    return cfg.get("base_model_name_or_path", BASE_MODEL_NAME)


def _validate_adapter_files(adapter_path):
    adapter_path = Path(adapter_path)
    st_path = adapter_path / "adapter_model.safetensors"
    if not st_path.exists():
        raise FileNotFoundError(f"Thiếu {st_path}")
    head = open(st_path, "rb").read(200)
    if head.startswith(b"version https://git-lfs"):
        raise RuntimeError(f"{st_path.name} là Git LFS pointer — chạy: git lfs pull")
    size_mb = st_path.stat().st_size / (1024 * 1024)
    if size_mb < 0.1:
        raise RuntimeError(f"{st_path.name} quá nhỏ ({size_mb:.2f} MB) — corrupt.")
    from safetensors import safe_open
    with safe_open(str(st_path), framework="pt", device="cpu") as f:
        _ = f.keys()
    print(f"Adapter file OK: {st_path.name} ({size_mb:.1f} MB)")


def load_eval_model(adapter_path, max_seq_length):
    import torch
    from unsloth import FastLanguageModel
    from peft import PeftModel

    _validate_adapter_files(adapter_path)
    base_model = _read_adapter_base_model(adapter_path)
    print(f"Eval load: base={base_model} | adapter={adapter_path}", flush=True)

    load_dtype = torch.float16 if not (LOAD_IN_4BIT or LOAD_IN_8BIT) else None
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model,
        max_seq_length=max_seq_length,
        dtype=load_dtype,
        load_in_4bit=LOAD_IN_4BIT,
        load_in_8bit=LOAD_IN_8BIT,
    )
    print("[Eval] Base loaded.", flush=True)

    model = PeftModel.from_pretrained(model, adapter_path, is_trainable=False, torch_device="cpu")
    if not hasattr(model, "peft_config") or not model.peft_config:
        raise RuntimeError("LoRA adapter KHÔNG được gắn.")
    print(f"Adapter OK: {list(model.peft_config.keys())}", flush=True)

    if USE_UNSLOTH_FOR_INFERENCE:
        FastLanguageModel.for_inference(model)

    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    print("[Eval] Model ready.", flush=True)
    return model, tokenizer


def _infer_predictions(method_name, adapter_path, samples, max_seq_length, *, log_every=50, desc=None):
    import time
    from tqdm.auto import tqdm

    clear_gpu()
    model_eval, tokenizer_eval = load_eval_model(adapter_path, max_seq_length)
    model_eval.generation_config.max_length = None
    model_eval.generation_config.max_new_tokens = MAX_NEW_TOKENS

    total = len(samples)
    desc = desc or f"Infer-{method_name}"
    print(f"[Infer] {method_name}: {total} samples (max_seq={max_seq_length})", flush=True)

    preds, t0 = [], time.time()
    pbar = tqdm(samples, total=total, desc=desc, bar_format=TQDM_BAR, file=sys.stdout, mininterval=1.0)
    for i, s in enumerate(pbar, 1):
        msgs = build_messages(s["context"], s["question"], for_inference=True)
        prompt = tokenizer_eval.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer_eval(
            prompt, return_tensors="pt", truncation=True, max_length=max_seq_length,
        ).to(model_eval.device)

        with torch.no_grad():
            out = model_eval.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs.get("attention_mask"),
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer_eval.pad_token_id,
                eos_token_id=tokenizer_eval.eos_token_id,
            )
        raw = tokenizer_eval.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, s["context"]) if USE_SPAN_POSTPROCESS else pred_raw

        preds.append({
            "id": s["id"],
            "uit_id": s["uit_id"],
            "prediction_raw": pred_raw,
            "prediction": pred,
        })

        if i == 1 or i % log_every == 0 or i == total:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 0.001)
            eta = (total - i) / max(rate, 0.001)
            print(f"[Infer] {i}/{total} | {elapsed/60:.1f}m | ETA {eta/60:.1f}m | {rate:.2f} sample/s", flush=True)
    pbar.close()
    print(f"[Infer] Done {total} in {(time.time()-t0)/60:.1f} min", flush=True)

    del model_eval, tokenizer_eval
    clear_gpu()
    return preds


def predictions_to_submission_dict(predictions):
    # uit_id -> answer text; sentinel no-answer -> ""
    out = {}
    for row in predictions:
        key = row.get("uit_id")
        if not key:
            raise KeyError(f"Prediction thiếu uit_id: {row}")
        pred = (row.get("prediction") or "").strip()
        out[key] = "" if is_no_answer(pred) else pred
    return out


def export_submission_one_adapter(method_name, adapter_dir, samples, max_seq_length):
    print(f"\n--- Submission export: {method_name} | {adapter_dir} ---", flush=True)
    if SUBMISSION_MAX_SAMPLES:
        samples = samples[:SUBMISSION_MAX_SAMPLES]
        print(f"[Submit] DEBUG: {len(samples)} câu", flush=True)
    else:
        print(f"[Submit] Full test: {len(samples)} câu -> results.json", flush=True)

    preds = _infer_predictions(
        method_name, adapter_dir, samples, max_seq_length,
        log_every=SUBMISSION_LOG_EVERY, desc=f"Submit-{method_name}",
    )
    results = predictions_to_submission_dict(preds)
    if len(results) != len(samples):
        raise RuntimeError(f"{method_name}: results.json thiếu câu ({len(results)} != {len(samples)})")

    out_path = Path(adapter_dir) / "results.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)
    n_empty = sum(1 for v in results.values() if v == "")
    print(f"[Submit] Saved {out_path} | total={len(results)} | noAns={n_empty} | hasAns={len(results)-n_empty}")
    return out_path

In [ ]:
# ========== Auto-detect adapter folders + Export ==========
def adapter_available(adapter_dir):
    d = Path(adapter_dir)
    return d.is_dir() and (d / "adapter_config.json").exists() and (d / "adapter_model.safetensors").exists()


print("Quét adapter trong thư mục hiện tại:", Path.cwd())
found = {}
for method in EXPORT_METHODS:
    adapter_dir = ADAPTER_SUBMISSION_DIRS.get(method)
    if adapter_dir and adapter_available(adapter_dir):
        found[method] = adapter_dir
        print(f"  [FOUND] {method:8s} -> {adapter_dir}")
    else:
        print(f"  [MISS ] {method:8s} -> {adapter_dir} (không có adapter, bỏ qua)")

if not found:
    raise RuntimeError("Không tìm thấy adapter nào hợp lệ. Kiểm tra tên folder / đã train chưa.")

written = {}
for method, adapter_dir in found.items():
    written[method] = export_submission_one_adapter(method, adapter_dir, test_samples, MAX_SEQ_LENGTH)

print("\n==== DONE ====")
for method, path in written.items():
    print(f"  {method}: {path}")